# 03 · ONNX 流式导出 + Colab 侧评测

两件事：

1. **导出流式 ONNX** 并做三层一致性校验 —— 本项目工程含量最高的一环
2. **算 PESQ / DNSMOS** —— PESQ 在 Windows 上装不上（`docs/ISSUES.md` I-05），
   只能在这边算

**关于流式导出**：ONNX 图是无状态的，而流式推理必然有状态（卷积缓存 + GRU 隐状态）。
唯一的办法是把状态显式外置成图的输入输出：

```
(当前帧, cache_0..4, gru_h) → (增强帧, new_cache_0..4, new_gru_h)
```

导出后**必须**验证「ONNX 流式 == PyTorch 整段」。只验「ONNX 流式 == PyTorch 流式」
是不够的 —— 如果 PyTorch 的流式实现本身就偷看了未来帧，两边会**一致地错**，
而离线指标依然好看。

In [ ]:
# ── 挂载 Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  配置 —— 所有路径与开关集中在这一个 cell，别处不要再写死路径
# ═══════════════════════════════════════════════════════════════════════

# Drive 上的项目根。**持久**，会话结束不丢。
DRIVE_ROOT = '/content/drive/MyDrive/Audio AI/RTSE'

# Colab 本地盘。**临时**，会话结束即消失，但读写比 Drive 快得多。
WORK_ROOT = '/content/rtse_work'

# ── 语料怎么放 ─────────────────────────────────────────────────────────
#   'hybrid' —— **推荐**。压缩包缓存在 Drive，每会话解压到本地盘。
#               一次下载永久有效；训练读取走本地盘全速。
#   'local'  —— 全在临时盘，用完即删，每个新会话都要重下。
DATA_MODE = 'hybrid'

# ── 数据规模 ───────────────────────────────────────────────────────────
# DNS5 干净语音的 split 切片数。每片 5.24 GB，实测约 **19 小时**，
# 落在"20~30 小时可管理子集"这个目标区间内。
# 切片档解压到末尾会报 EOF，属正常（详见 fetch_dns 的说明）。
N_SPEECH_SHARDS = 1
# DNS 噪声分片：audioset（日常环境声）+ freesound（标注音效）各取几片。
N_AUDIOSET_SHARDS = 2
N_FREESOUND_SHARDS = 1

# ── 快速验证模式 ───────────────────────────────────────────────────────
# True = 跳过 DNS 大文件，只下 WenetSpeech（约 520 MB）验证整条链路。
QUICK_TEST = False

# ═══════════════════════════════════════════════════════════════════════

import os, sys, json, shutil, subprocess, time
from pathlib import Path
from shlex import quote as shq          # 路径里有空格时，所有 shell 命令都靠它

assert DATA_MODE in ('hybrid', 'local'), 'DATA_MODE 只能是 hybrid / local'

DRIVE = DRIVE_ROOT
WORK = WORK_ROOT
ARCHIVE_DIR = f'{WORK}/archives' if DATA_MODE == 'local' else f'{DRIVE}/archives'
DATA = f'{WORK}/data'
KEEP_ARCHIVE = DATA_MODE != 'local'

CKPT_DIR = f'{DRIVE}/checkpoints'   # 训练断点，每 epoch 保存
MODEL_DIR = f'{DRIVE}/models'       # 导出的 ONNX
TESTSET_DIR = f'{DRIVE}/testset'    # 固定测试集
LOG_DIR = f'{DRIVE}/logs'

assert os.path.isdir(DRIVE), (
    f'Drive 上找不到 {DRIVE}\n'
    '检查：① Drive 已挂载成功；② DRIVE_ROOT 与你实际的目录一致（区分大小写，空格照写）。'
)
for d in [WORK, ARCHIVE_DIR, DATA, CKPT_DIR, MODEL_DIR, TESTSET_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print('目录布局')
print('─' * 74)
print(f'  代码包(需手动上传)  {DRIVE}/rtse-colab.zip')
print(f'  压缩包缓存          {ARCHIVE_DIR}')
print(f'  语料解压目标        {DATA}')
print(f'  数据清单            {DRIVE}/manifest.json')
print(f'  固定测试集          {TESTSET_DIR}')
print(f'  训练断点  ★         {CKPT_DIR}/<模型名>/{{last,best}}.pt')
print(f'  导出模型  ★         {MODEL_DIR}/<模型名>.onnx')
print('─' * 74)
print(f'  数据模式  {DATA_MODE}')
print(f'  语料      {"快速验证(仅 WenetSpeech)" if QUICK_TEST else "完整(DNS 英文训练 + WenetSpeech 中文评测)"}')
print()

!df -h /content | tail -1

In [ ]:
# ── 安装项目代码 ────────────────────────────────────────────────────────
# rtse-colab.zip 由本地 `uv run python scripts/pack_for_colab.py` 生成，
# 需要手动上传到 DRIVE_ROOT 目录下。**代码改过就要重新上传**，
# 否则 Colab 跑的还是旧逻辑（这个坑踩过，见 docs/ISSUES.md）。
ZIP = f'{DRIVE}/rtse-colab.zip'
assert os.path.exists(ZIP), (
    f'找不到 {ZIP}\n'
    '请先在本地执行 `uv run python scripts/pack_for_colab.py`，'
    f'再把 dist/rtse-colab.zip 上传到 Drive 的 {DRIVE} 下。\n'
    f'该目录下现有：{sorted(os.listdir(DRIVE))[:12]}'
)

SRC = f'{WORK}/rtse-src'
shutil.rmtree(SRC, ignore_errors=True)
os.makedirs(SRC, exist_ok=True)
!unzip -q -o {shq(ZIP)} -d {shq(SRC)}

# 只装项目需要而 Colab 没预装的。不用 `pip install -e .`：那会去解析 pyproject
# 里锁定的 torch CPU 索引，把 Colab 自带的 GPU 版 torch 覆盖掉，训练慢几十倍。
!pip install -q soxr pystoi jiwer webrtcvad-wheels pesq onnx onnxruntime opencc-python-reimplemented 2>&1 | tail -2

sys.path.insert(0, f'{SRC}/src')
import rtse
print('rtse', rtse.__version__, '| SR', rtse.SAMPLE_RATE, '| n_fft', rtse.N_FFT, '| hop', rtse.HOP_LENGTH)

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. 导出并校验

In [ ]:
import torch
from rtse.models import build_model
from rtse.train.export import export_streaming_onnx

MODELS = ['crn-nano', 'crn-lite']
export_info = {}

for name in MODELS:
    ck = Path(f'{CKPT_DIR}/{name}/best.pt')
    if not ck.exists():
        print(f'[skip] {name}: 没有 best.pt，先跑 02_train.ipynb'); continue

    state = torch.load(ck, map_location='cpu', weights_only=False)
    model = build_model(name)
    model.load_state_dict(state['model'])
    model.eval()

    info = export_streaming_onnx(model, f'{MODEL_DIR}/{name}.onnx', verify=True)
    v = info['verification']
    export_info[name] = info

    print(f'\n=== {name} ===')
    print(f"  训练到 epoch {state['epoch']}，最佳 val SI-SDR {-state['best_val']:.3f} dB")
    print(f"  参数 {info['params']:,}   文件 {info['size_kb']} KB")
    print(f"  ONNX流式 vs PyTorch整段 : {v['onnx_vs_pytorch_batch']:.3e} (相对 {v['relative_error']:.3e})")
    print(f"  PyTorch流式 vs 整段     : {v['pytorch_streaming_vs_batch']:.3e}")
    print(f"  状态形状稳定            : {v['state_shape_stable']}")
    print(f"  ==> {'PASS ✅' if v['passed'] else 'FAIL ❌ 不要下载这个模型，先查因果性'}")

Path(f'{MODEL_DIR}/export_info.json').write_text(
    json.dumps(export_info, ensure_ascii=False, indent=1), encoding='utf-8')

## 2. 下载 DNSMOS 模型

DNSMOS 是**无参考** MOS 预测，是实时麦克风演示里唯一能显示的质量指标
（那里没有干净参考）。模型来自微软 DNS-Challenge 仓库，只有几 MB。

In [ ]:
os.makedirs(f'{MODEL_DIR}/dnsmos', exist_ok=True)
!wget -q -O "{MODEL_DIR}/dnsmos/sig_bak_ovr.onnx" \
  https://raw.githubusercontent.com/microsoft/DNS-Challenge/master/DNSMOS/DNSMOS/sig_bak_ovr.onnx \
  && ls -lh "{MODEL_DIR}/dnsmos/"

# 若 404，去 https://github.com/microsoft/DNS-Challenge 的 DNSMOS 目录确认最新路径。
# 拿不到也不影响主流程：本地评测会自动把 DNSMOS 列标 n/a。

## 3. 在测试集上评测（含 PESQ）

按 `noise_kind`（稳态/非稳态）和 `rir_kind`（合成/真实）分组汇总 ——
这两个维度正是这套数据设计要回答的核心问题。

In [ ]:
import numpy as np
from tqdm.auto import tqdm
from rtse.audio.io import read_audio
from rtse.metrics.intrusive import si_sdr, stoi, estoi, pesq, seg_snr
from rtse.runtime import Pipeline, OnnxEnhancer
from rtse.dsp import build_dsp
from rtse.vad import build_vad

idx = json.loads(Path(f'{TESTSET_DIR}/index.json').read_text(encoding='utf-8'))
records = idx['records']
print(f'测试集 {len(records)} 个样本')

METHODS = ['none', 'specsub', 'wiener', 'mmse-lsa'] + list(export_info)

def make(method):
    if method == 'none': return None
    if method in ('specsub', 'wiener', 'mmse-lsa'): return build_dsp(method)
    return OnnxEnhancer(f'{MODEL_DIR}/{method}.onnx')

rows = []
for method in METHODS:
    enh = make(method)
    pipe = Pipeline(enhancer=enh, vad=build_vad('energy'))
    for r in tqdm(records, desc=f'{method:>10}', leave=False):
        clean = read_audio(f'{TESTSET_DIR}/{r["clean"]}')
        noisy = read_audio(f'{TESTSET_DIR}/{r["noisy"]}')
        pipe.reset()
        out, _ = pipe.process_signal(noisy)
        rows.append({'id': r['id'], 'method': method, 'snr': r['snr'],
                     'noise_kind': r['noise_kind'], 'rir_kind': r['rir_kind'],
                     'rt60_measured': r['rt60_measured'],
                     'si_sdr': si_sdr(clean, out), 'seg_snr': seg_snr(clean, out),
                     'stoi': stoi(clean, out), 'estoi': estoi(clean, out),
                     'pesq': pesq(clean, out)})

Path(f'{DRIVE}/colab_metrics.json').write_text(json.dumps(rows, ensure_ascii=False), encoding='utf-8')
print(f'已写入 {len(rows)} 条指标 → {DRIVE}/colab_metrics.json')

In [ ]:
# 汇总（正式的表在本地 rtse-eval 里出，这里只是快速看一眼）
import collections, statistics as st
print(f"{'method':<12}{'noise':<15}{'rir':<8}{'SI-SDR':>9}{'STOI':>8}{'PESQ':>8}")
print('-' * 60)
agg = collections.defaultdict(list)
for r in rows:
    agg[(r['method'], r['noise_kind'], r['rir_kind'])].append(r)
for m in METHODS:
    for nk in ['stationary', 'nonstationary']:
        for rk in ['synth', 'real']:
            g = agg.get((m, nk, rk))
            if not g: continue
            pq = [r['pesq'] for r in g if r['pesq'] is not None]
            print(f"{m:<12}{nk:<15}{rk:<8}"
                  f"{st.mean(r['si_sdr'] for r in g):>9.2f}"
                  f"{st.mean(r['stoi'] for r in g):>8.3f}"
                  f"{(st.mean(pq) if pq else float('nan')):>8.3f}")

## 4. 打包回传

| Drive 上的文件 | 放到本地 |
|---|---|
| `models/*.onnx` | `models/` |
| `models/dnsmos/sig_bak_ovr.onnx` | `models/dnsmos/` |
| `colab_metrics.json` | `results/` |
| `testset.zip`（01 生成） | 解压到 `data/`，使 `data/testset/index.json` 存在 |

放好后本地 `uv run rtse-doctor` 应该全绿，`uv run rtse-eval` 可以跑完整评测（含 CER）。

In [ ]:
!cd "{DRIVE}" && rm -f colab_outputs.zip && zip -q -r colab_outputs.zip models colab_metrics.json && ls -lh colab_outputs.zip
print(f'从 Google Drive 下载 {DRIVE}/colab_outputs.zip 即可。')